In [ ]:
import zipfile
import os
import shutil
import tempfile
from google.colab import files
from collections import defaultdict

def rename_in_zip(zip_path: str, prefix_mapping: dict, output_name: str):
    """
    Renombra archivos PDF dentro de un ZIP cambiando SOLO el prefijo si coincide
    con alguna de las claves de prefix_mapping y conserva el resto del nombre.
    Excepción: si existen tanto 'EPI' como 'HEV' en el mismo ZIP, no renombrar EPI.
    ADEMÁS: Crea DOS archivos ZIP separados:
      - {output_name}_SOPORTE.zip: archivos .pdf
      - {output_name}_RIPS.zip: archivos .json y .xml
    """
    sorted_keys = sorted(prefix_mapping.keys(), key=len, reverse=True)

    renamed_count = 0
    soporte_count = 0
    rips_count = 0
    conflicts = []

    # Contadores de diagnóstico
    total_pdf_original = 0
    total_json_xml_original = 0

    with tempfile.TemporaryDirectory() as tmpdir:
        # 1) Extraer ZIP y contar archivos originales
        print("📦 Extrayendo archivos del ZIP...")
        with zipfile.ZipFile(zip_path, 'r') as zin:
            zin.extractall(tmpdir)

            # Contar archivos en el ZIP original
            for item in zin.namelist():
                if not item.endswith('/'):  # Ignorar carpetas
                    ext = os.path.splitext(item)[1].lower()
                    if ext == '.pdf':
                        total_pdf_original += 1
                    elif ext in ['.json', '.xml']:
                        total_json_xml_original += 1

        print(f"   📊 PDFs encontrados: {total_pdf_original}")
        print(f"   📊 JSON/XML encontrados: {total_json_xml_original}")

        # Crear carpetas temporales
        soporte_dir = os.path.join(tmpdir, 'SOPORTE')
        rips_dir = os.path.join(tmpdir, 'RIPS')
        os.makedirs(soporte_dir, exist_ok=True)
        os.makedirs(rips_dir, exist_ok=True)

        # Recolectar TODOS los archivos primero
        all_files = []
        for root, _, files_ in os.walk(tmpdir):
            # Evitar procesar las carpetas que acabamos de crear
            if root.startswith(soporte_dir) or root.startswith(rips_dir):
                continue
            for fn in files_:
                all_files.append(os.path.join(root, fn))

        # Detectar si coexisten archivos con prefijo HEV y EPI
        has_epi = False
        has_hev = False
        for file_path in all_files:
            fn = os.path.basename(file_path)
            name, ext = os.path.splitext(fn)
            if ext.lower() == ".pdf":
                if name.startswith("EPI"):
                    has_epi = True
                if name.startswith("HEV"):
                    has_hev = True

        skip_epi = has_epi and has_hev
        if skip_epi:
            print("⚠️  Detectados prefijos EPI y HEV - EPI no será renombrado")

        # Diccionario para detectar nombres duplicados
        used_names = defaultdict(list)

        # 2) Procesar TODOS los archivos recolectados
        for file_path in all_files:
            fn = os.path.basename(file_path)
            name, ext = os.path.splitext(fn)

            # Archivos PDF -> SOPORTE
            if ext.lower() == ".pdf":
                new_name = fn
                was_renamed = False

                # Si se cumple la excepción: NO renombrar EPI
                if not (skip_epi and name.startswith("EPI")):
                    # Renombrar según prefix_mapping
                    for old_pref in sorted_keys:
                        if name.startswith(old_pref):
                            new_pref = prefix_mapping[old_pref]
                            rest = name[len(old_pref):]
                            new_name = f"{new_pref}{rest}{ext}"
                            print(f"  ✏️  {fn} → {new_name}")
                            renamed_count += 1
                            was_renamed = True
                            break

                if not was_renamed:
                    print(f"  📄 {fn} (sin cambios)")

                new_path = os.path.join(soporte_dir, new_name)

                # Detectar conflictos de nombres
                if os.path.exists(new_path):
                    conflicts.append(f"⚠️ CONFLICTO: {new_name} ya existe")
                    # Agregar sufijo para evitar sobrescritura
                    base, extension = os.path.splitext(new_name)
                    counter = 1
                    while os.path.exists(new_path):
                        new_name = f"{base}_DUPLICADO{counter}{extension}"
                        new_path = os.path.join(soporte_dir, new_name)
                        counter += 1
                    conflicts.append(f"   → Renombrado como: {new_name}")

                used_names[new_name].append(fn)
                shutil.copy2(file_path, new_path)  # Usar copy2 en vez de move
                soporte_count += 1

            # Archivos JSON y XML -> RIPS
            elif ext.lower() in ['.json', '.xml']:
                new_path = os.path.join(rips_dir, fn)

                if os.path.exists(new_path):
                    conflicts.append(f"⚠️ CONFLICTO: {fn} ya existe en RIPS")

                shutil.copy2(file_path, new_path)
                print(f"  📋 {fn} → RIPS/")
                rips_count += 1

        # 3) Validación final
        print(f"\n{'='*60}")
        print("📊 RESUMEN DE PROCESAMIENTO:")
        print(f"{'='*60}")
        print(f"PDFs en ZIP original:     {total_pdf_original}")
        print(f"PDFs procesados:          {soporte_count}")
        print(f"PDFs renombrados:         {renamed_count}")
        print(f"JSON/XML en ZIP original: {total_json_xml_original}")
        print(f"JSON/XML procesados:      {rips_count}")

        if soporte_count != total_pdf_original:
            print(f"\n⚠️  ADVERTENCIA: Faltan {total_pdf_original - soporte_count} PDF(s)")
        else:
            print(f"\n✅ Todos los PDFs fueron procesados correctamente")

        if rips_count != total_json_xml_original:
            print(f"⚠️  ADVERTENCIA: Faltan {total_json_xml_original - rips_count} archivo(s) JSON/XML")
        else:
            print(f"✅ Todos los archivos JSON/XML fueron procesados correctamente")

        # Mostrar conflictos si existen
        if conflicts:
            print(f"\n{'='*60}")
            print("⚠️  CONFLICTOS DETECTADOS:")
            print(f"{'='*60}")
            for conflict in conflicts:
                print(conflict)

        # 4) Crear DOS archivos ZIP separados
        base_name = output_name.replace('.zip', '')

        # ZIP de SOPORTE
        soporte_zip = f"{base_name}_SOPORTE.zip"
        shutil.make_archive(soporte_zip.replace('.zip', ''), 'zip', soporte_dir)
        print(f"\n✅ {soporte_zip} creado ({soporte_count} archivos)")

        # ZIP de RIPS
        rips_zip = f"{base_name}_RIPS.zip"
        shutil.make_archive(rips_zip.replace('.zip', ''), 'zip', rips_dir)
        print(f"✅ {rips_zip} creado ({rips_count} archivos)")

        return soporte_zip, rips_zip

# Ejecución
print("="*60)
print("📤 SUBE TUS ARCHIVOS ZIP")
print("="*60)
uploaded = files.upload()

prefix_mapping = {
    'SI': 'FEV_900472731_SI',
    #'PDX': 'HEV',
    #'PDE': 'OPF',
    #'OPF': 'PDE'
}

print(f"\n🔄 Procesando {len(uploaded)} archivo(s)...\n")

for fname in uploaded:
    print(f"\n{'━'*60}")
    print(f"📁 PROCESANDO: {fname}")
    print(f"{'━'*60}\n")

    soporte_zip, rips_zip = rename_in_zip(fname, prefix_mapping, fname)

    # Descargar ambos ZIPs por separado
    print(f"\n⬇️  Descargando archivos...")
    files.download(soporte_zip)
    files.download(rips_zip)
    print(f"✅ Descarga completada\n")

📤 SUBE TUS ARCHIVOS ZIP
